# 🔧 Příprava featur — INSTRUKTORSKÝ notebook

**Spustit JEDNOU před akcí, ideálně na Google Colab s GPU**
(*Runtime → Change runtime type → GPU*).

Notebook stáhne dataset malárie, prožene všech 27 558 obrázků zmraženým **ResNet50**,
uloží výsledné **featury** (2048 čísel na buňku) a rozdělí je na `train` / `val` / `test`.
Tyhle soubory pak nahrajete na GitHub Release a studenti je jen stahují.

Výstup:
- `train.npz`, `val.npz` — featury **+ labely** (pro studenty)
- `test_features.npz` — featury **bez labelů** (pro studenty, na odevzdání)
- `test_labels.npz` — labely testu (**necháte si**, na vyhodnocení soutěže)
- `samples.zip` — pár ukázkových obrázků do studentského notebooku

Běh na Colab T4 trvá řádově **5–10 minut**.

In [ ]:
# kagglehub bývá na Colabu předinstalovaný; pro jistotu:
!pip -q install kagglehub

import os, glob, zipfile, shutil
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
print("zařízení:", device, "| GPU:", torch.cuda.get_device_name(0) if device == "cuda" else "—")

## 1 · Stažení datasetu

Stáhneme Kaggle dataset `iarunava/cell-images-for-detecting-malaria`. Sada má dvě složky
`Parasitized/` a `Uninfected/`, ale historicky obsahuje i **vnořenou složku** a zatoulané
ne-obrázkové soubory (`Thumbs.db`, `*.db`). Proto sebereme rekurzivně **jen `*.png`** a label
určíme z názvu nadřazené složky.

In [ ]:
import kagglehub
root = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
print("staženo do:", root)

png = glob.glob(os.path.join(root, "**", "*.png"), recursive=True)
items = []
for p in png:
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "parasitized" in parent:
        items.append((p, 1))
    elif "uninfected" in parent:
        items.append((p, 0))
# DETERMINISTICKÉ pořadí (nezávislé na stroji) → stejný split všude, sedí na test_labels.npz
items.sort(key=lambda t: (os.path.basename(t[0]), t[1]))
print("nalezeno obrázků:", len(items),
      "| napadených:", sum(l for _, l in items),
      "| zdravých:", sum(1 - l for _, l in items))

## 2 · Předzpracování a DataLoader

Každý obrázek zmenšíme na 224×224 a znormalizujeme přesně tak, jak to očekává ResNet
předtrénovaný na ImageNetu (proto ty konkrétní průměry a odchylky).

In [ ]:
prep = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BunkyDataset(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label

loader = DataLoader(BunkyDataset(items, prep), batch_size=128,
                    shuffle=False, num_workers=2)
print("dávek:", len(loader))

## 3 · Encoder ResNet50 → featury

Načteme ResNet50 s vahami z ImageNetu, **odřízneme poslední klasifikační vrstvu**
(`fc = Identity`) a necháme ho v režimu `eval()` bez počítání gradientů. Výstupem je pro
každou buňku **2048-rozměrný vektor**.

In [ ]:
encoder = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
encoder.fc = nn.Identity()
encoder.eval().to(device)

feats, labels = [], []
with torch.no_grad():
    for k, (xb, yb) in enumerate(loader):
        f = encoder(xb.to(device)).cpu().numpy().astype(np.float16)
        feats.append(f); labels.append(np.asarray(yb))
        if k % 20 == 0:
            print(f"  zpracováno {k*128:6d} / {len(items)}")
X = np.concatenate(feats).astype(np.float16)
y = np.concatenate(labels).astype(np.int64)
print("hotovo. featury:", X.shape, "| labely:", y.shape, "| dtype:", X.dtype)

## 4 · Stratifikovaný split 70 / 15 / 15

Rozdělíme **stratifikovaně** (zachová poměr tříd) a s **pevným seedem**, aby oba týmy
pracovaly s naprosto stejnými daty — soutěž musí být férová.

In [ ]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=42)  # 0.1765*0.85 ≈ 0.15

for name, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s}: {len(yy):6d}  (napadených {int(yy.sum())}, zdravých {int((1-yy).sum())})")

np.savez_compressed("train.npz", X=X_train.astype(np.float16), y=y_train)
np.savez_compressed("val.npz",   X=X_val.astype(np.float16),   y=y_val)
np.savez_compressed("test_features.npz", X=X_test.astype(np.float16))   # BEZ labelů
np.savez_compressed("test_labels.npz",   y=y_test)                       # jen pro vás!
for f in ["train.npz", "val.npz", "test_features.npz", "test_labels.npz"]:
    print(f, "→", round(os.path.getsize(f)/1e6, 1), "MB")

## 5 · Ukázkové obrázky pro studenty

Zkopírujeme pár desítek obrázků (zvlášť napadené a zdravé), aby si je studenti mohli
prohlédnout v sekci 2 svého notebooku.

In [ ]:
os.makedirs("samples/Parasitized", exist_ok=True)
os.makedirs("samples/Uninfected", exist_ok=True)
paras = [p for p, l in items if l == 1][:15]
heal  = [p for p, l in items if l == 0][:15]
for p in paras: shutil.copy(p, "samples/Parasitized/")
for p in heal:  shutil.copy(p, "samples/Uninfected/")

with zipfile.ZipFile("samples.zip", "w") as z:
    for f in glob.glob("samples/**/*.png", recursive=True):
        z.write(f)
print("samples.zip →", round(os.path.getsize("samples.zip")/1e6, 2), "MB")

## 6 · Kontrola a hosting

Rychlá kontrola, že soubory jdou znovu načíst a mají správné tvary. Pak je nahrajte:

**Doporučeno — GitHub Release:**
1. Vytvořte (klidně prázdný) repozitář, např. `malaria-veletrh`.
2. *Releases → Draft a new release*, tag `data-v1`.
3. Přetáhněte sem `train.npz`, `val.npz`, `test_features.npz`, `samples.zip`
   (`test_labels.npz` **NE** — ten si necháte!).
4. Publish. URL assetu bude `https://github.com/<vy>/<repo>/releases/download/data-v1/train.npz`.
5. Do studentského notebooku doplňte `DATA_BASE_URL` na
   `https://github.com/<vy>/<repo>/releases/download/data-v1`.

**Záloha — Google Drive:** nahrajte soubory, dejte „kdokoli s odkazem", a ve studentském
notebooku použijte `gdown <ID>` místo `wget`.

In [ ]:
for f in ["train.npz", "val.npz", "test_features.npz"]:
    d = np.load(f)
    print(f, "→ X", d["X"].shape, "| klíče:", list(d.keys()))
d = np.load("test_labels.npz")
print("test_labels.npz → y", d["y"].shape, "(zůstává jen u vás)")
print("\nVše připraveno k nahrání. 🎉")